# Porto Taxi — Popular Long Sub-Routes, Activity Zones & Anomalies

Interactive demo of the Spark pipeline's results, for **Colab Enterprise**.

It reads the small **result CSVs** produced by the mining jobs — top-100 routes
for each method at each length configuration, activity zones, anomalies — **not**
the raw 1.9 GB data, so it renders instantly.

Pipeline that produced these files (see `docs/DATAPROC.md`):
`clean_data → feature_engineering → spatial_encoding →
route_mining_{suffix_array, maximal, clustering, graph} → anomaly_analysis`.

**Methods**

| | method | how it finds routes |
|---|---|---|
| A | clustering | MinHash-LSH over directed bigram shingles → star clustering → the longest cell run its members share |
| B | maximal-frequent | contiguous n-gram support table → maximal among those clearing X%, X calibrated per length |
| C | transition graph | PageRank activity zones + dominant-flow heavy paths, validated against real trips |
| D | suffix array | generalised suffix array + LCP intervals — exact, without enumerating windows |

All four report the same unit: a contiguous sub-route with a distinct-trip support.

In [1]:
# 1. Dependencies. Idempotent: installs only what is missing, so this runs
#    unchanged locally (already installed) and in Colab (fresh VM).
import importlib.util, subprocess, sys

for spec, mod in [("folium==0.16.0", "folium"), ("h3==3.7.7", "h3"),
                  ("gcsfs", "gcsfs")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", spec],
                       check=False)

import pandas as pd, folium, h3
print(f"pandas {pd.__version__} | folium {folium.__version__} | h3 {h3.__version__}")

pandas 2.2.2 | folium 0.16.0 | h3 3.7.7


In [2]:
# 2. Where the result CSVs live.
#    Auto-detects: uses the local pipeline outputs when present (so the
#    notebook is runnable straight from a checkout), otherwise the bucket.
#    In Colab Enterprise, set BUCKET and it authenticates to GCS for you.
import os

BUCKET = 'gs://YOUR_BUCKET'          # <-- edit for the cloud run
SCALE  = 'full'                      # 'full' | 'mid' | 'sample'

LOCAL = next((p for p in ('outputs/routes', '../outputs/routes')
              if os.path.isdir(p)), None)
BASE = LOCAL if LOCAL else f'{BUCKET}/porto/outputs/routes'
print(f'reading from: {BASE}  (scale={SCALE})')

def load(name):
    try:
        return pd.read_csv(f'{BASE}/{name}_{SCALE}.csv')
    except Exception as e:
        print('skip', name, '->', type(e).__name__)
        return pd.DataFrame()

M = {'A': load('clustering_top100'),
     'B': load('maximal_frequent_top100'),
     'C': load('graph_heavy_paths_top100'),
     'D': load('suffix_array_top100')}
Z  = load('activity_zones')
AN = load('anomalies_top50')
print({k: len(v) for k, v in M.items()}, '| zones:', len(Z), '| anomalies:', len(AN))

reading from: ../outputs/routes  (scale=full)
skip maximal_frequent_top100 -> FileNotFoundError
{'A': 322, 'B': 0, 'C': 314, 'D': 420} | zones: 50 | anomalies: 50


In [3]:
# 3. Coverage: how many routes each method found at each length configuration.
#    An empty cell at >=20/40 km is a real finding, not a bug: it means no
#    corridor of that length is driven by enough distinct trips at this scale.
LENGTHS = [1, 3, 5, 10, 20, 40]
cov = pd.DataFrame(
    {m: {L: int((df['min_len_km'] == L).sum()) if len(df) else 0 for L in LENGTHS}
     for m, df in M.items()})
cov.index.name = 'min_len_km'
cov

,A,B,C,D
min_len_km,,,,
1,100,0,100,100
3,100,0,100,100
5,100,0,100,100
10,22,0,14,100
20,0,0,0,20
40,0,0,0,0


In [4]:
# 4. The map: every method x every length configuration as its own layer.
#    Use the layer control (top right) to switch between the six configurations.
DELIM = '>'
COLOURS = {'A': '#2e8b3d', 'B': '#1f5fbf', 'C': '#d1341c', 'D': '#7b2fbf'}
ROUTE_COL = {'A': 'subroute', 'B': 'subroute', 'C': 'route', 'D': 'subroute'}
LABEL = {'A': 'A clustering', 'B': 'B maximal-frequent',
         'C': 'C transition-graph', 'D': 'D suffix-array'}
SHOW_L = 3          # the band where every method has results

def polyline(cells):
    return [list(h3.h3_to_geo(c)) for c in str(cells).split(DELIM) if c]

m = folium.Map(location=(41.157, -8.629), zoom_start=12, tiles='cartodbpositron')

for key, df in M.items():
    if not len(df):
        continue
    for L in LENGTHS:
        sub = df[df.min_len_km == L].head(25)
        if not len(sub):
            continue
        fg = folium.FeatureGroup(name=f'{LABEL[key]} (>={L} km)', show=(L == SHOW_L))
        for _, r in sub.iterrows():
            pts = polyline(r[ROUTE_COL[key]])
            if len(pts) >= 2:
                folium.PolyLine(
                    pts, color=COLOURS[key], weight=3, opacity=0.7,
                    tooltip=f"{LABEL[key]} >={L}km | support={r['support']} "
                            f"| {r['length_km']:.2f} km").add_to(fg)
        fg.add_to(m)

if len(Z):
    zfg = folium.FeatureGroup(name='Activity zones (PageRank)', show=True)
    for _, z in Z.iterrows():
        folium.CircleMarker((z.lat, z.lon), radius=max(3, 14 - int(z['rank']) // 4),
                            color='#e8850c', fill=True, fill_opacity=0.5,
                            tooltip=f"zone #{z['rank']} pr={z['pagerank']:.5f}").add_to(zfg)
    zfg.add_to(m)

if len(AN):
    afg = folium.FeatureGroup(name='Anomalous routes', show=False)
    for _, a in AN.iterrows():
        folium.CircleMarker((a.start_lat, a.start_lon), radius=5, color='#555555',
                            fill=True, fill_opacity=0.7,
                            tooltip=f"score={a.anomaly_score} dist={a.dist_km}km").add_to(afg)
    afg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

In [5]:
# 5. The deliverable: top-100 popular long sub-routes for one config.
#    Method D (suffix array) is the default because it is the one that
#    carries the deliverable at full scale -- B is sample-scale only, so
#    hardcoding B here fails on a full-scale run.
METHOD = next((k for k in ('D', 'B', 'A', 'C') if len(M[k])), None)
L = 3   # one of 1, 3, 5, 10, 20, 40

if METHOD is None:
    print('no method produced routes; run the mining stages first')
else:
    df = M[METHOD]
    cols = [c for c in ['rank', 'support', 'support_taxis', 'length_km',
                        'n_cells', 'min_sup', 'x_pct'] if c in df.columns]
    print(f'Method {METHOD}, corridors of at least {L} km')
    display(df[df.min_len_km == L][cols].head(100))

Method D, corridors of at least 3 km


,rank,support,support_taxis,length_km,n_cells,min_sup,x_pct
100,1,4701,435,3.668,11,2500,0.15485
101,2,4350,421,3.263,10,2500,0.15485
102,3,4304,421,5.072,15,2500,0.15485
103,4,4115,405,4.409,13,2500,0.15485
104,5,4105,376,4.216,12,2500,0.15485
...,...,...,...,...,...,...,...
195,96,2905,399,4.388,13,2500,0.15485
196,97,2886,377,3.260,9,2500,0.15485
197,98,2877,312,3.264,10,2500,0.15485
198,99,2876,346,3.284,10,2500,0.15485


In [6]:
# 6. Do independent methods agree? Overlap of the cell sets they report at >=3 km.
#    Two methods with different failure modes converging on the same corridors is
#    the strongest evidence available that those corridors are real.
def cellsets(key, L=3):
    df = M[key]
    if not len(df):
        return []
    return [frozenset(str(r[ROUTE_COL[key]]).split(DELIM))
            for _, r in df[df.min_len_km == L].iterrows()]

def match_fraction(xs, ys, thr=0.5):
    if not xs or not ys:
        return float('nan')
    hit = sum(any(len(x & y) / len(x | y) >= thr for y in ys) for x in xs)
    return hit / len(xs)

present = [k for k in M if len(M[k])]
S = {k: cellsets(k) for k in present}
pd.DataFrame({c: {r: (1.0 if r == c else match_fraction(S[r], S[c])) for r in present}
              for c in present}).round(2)

,A,C,D
A,1.00,0.47,0.93
C,0.31,1.00,0.42
D,0.84,0.47,1.00
